# Аналитика и визуализация начислений ЖКУ (ЕПД МосОблЕИРЦ)

Данный ноутбук содержит примеры аналитических SQL-запросов к оцифрованным квитанциям из базы данных SQLite `db/receipts.db` и построение **интерактивных графиков Plotly** с детальными подсказками (tooltips):
- 🏢 **Паспорт помещения и многоквартирного дома** (площадь квартиры, площади дома и МОП, доля в МКД)
- 📊 **Динамика общей суммы платежей и стоимости 1 кв.м** (Bar Chart с подсказками руб/м²)
- 🥧 **Структура расходов по категориям** (Donut Chart с процентами и суммами в рублях)
- 🌡️ **Сезонный анализ отопления** (двухосевой график: расход в Гкал и сумма начислений в рублях)
- ⚡ **Потребление ресурсов** (независимые шкалы для электроэнергии в кВт⋅ч и воды в м³)
- 🏆 **Рейтинг самых затратных услуг** за весь наблюдаемый период (горизонтальная диаграмма)
- 🥇 **Топ-3 самых затратных услуг по годам** (панели по годам с долями в % от годового бюджета)
- 📈 **Распределение категорий услуг в разрезе годов** (абсолютные рубли + 100% нормированные доли)
- 🔄 **Годовая динамика Year-over-Year** (сезонные кривые, дельта в ₽ и %, средний чек)
- 🏷️ **Анализ индексации тарифов (Tariff Tracker)** (нормализованный индекс цен и даты повышения)
- ⚖️ **Факторный анализ «Тариф vs Объем»** (декомпозиция: вклад потребления vs вклад цен)
- 🎛️ **Интерактивный селектор услуг** (детальный разбор любой услуги из выпадающего списка)
- 🔍 **Сведения о перерасчетах** (выборка корректировок за ОДН/КР СОИ)


In [79]:
import sqlite3
from pathlib import Path

# Проверяем наличие библиотек pandas, plotly и matplotlib
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("Для работы с табличными данными установите pandas: pip install pandas")

try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    from plotly.basedatatypes import BaseFigure
    from IPython.display import display, HTML
    
    pio.templates.default = "plotly_white"
    HAS_PLOTLY = True
    
    # Гарантируем чистую ссылку на оригинальный BaseFigure.show (защита от зацикливания при перезапуске ячейки)
    if not hasattr(BaseFigure, "_unwrapped_show"):
        BaseFigure._unwrapped_show = BaseFigure.show
    
    def _safe_fig_show(self, *args, **kwargs):
        try:
            return BaseFigure._unwrapped_show(self, *args, **kwargs)
        except Exception as e:
            if "nbformat" in str(e).lower() or "mime" in str(e).lower():
                return display(HTML(self.to_html(include_plotlyjs="cdn", full_html=False)))
            raise e
    go.Figure.show = _safe_fig_show
    
    def show_fig(fig):
        """Универсальный вывод графиков Plotly в Jupyter, VS Code и PyCharm без сбоев из-за nbformat."""
        try:
            return BaseFigure._unwrapped_show(fig)
        except Exception as e:
            if "nbformat" in str(e).lower() or "mime" in str(e).lower():
                display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))
            else:
                raise e
except ImportError:
    HAS_PLOTLY = False
    print("⚠️ Для отображения интерактивных графиков установите plotly: pip install plotly nbformat")

try:
    import matplotlib.pyplot as plt
    import matplotlib.ticker as ticker
    plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

HAS_PLOT_LIBS = HAS_PANDAS and (HAS_PLOTLY or HAS_MPL)

# Путь к базе данных receipts.db (с автоопределением рабочей директории)
DB_PATH = Path("..") / "db" / "receipts.db"
if not DB_PATH.exists():
    DB_PATH = Path("db") / "receipts.db"

conn = sqlite3.connect(str(DB_PATH))
conn.row_factory = sqlite3.Row
print(f"Подключено к базе данных: {DB_PATH.resolve()}")


Подключено к базе данных: D:\PyProjects\_my\pdf_table_parcer\db\receipts.db


---
## 1. Лицевой счет, паспорт помещения и реестр квитанций
Сводные характеристики жилого помещения (площадь квартиры, количество проживающих), параметры многоквартирного дома (общая площадь МКД, жилые помещения, места общего пользования) и расчетная доля в общедомовом имуществе.


In [80]:
cursor = conn.cursor()
account = cursor.execute("SELECT * FROM accounts LIMIT 1").fetchone()
if account:
    print("=" * 75)
    print(f"👤 Лицевой счет: {account['account_number']}")
    print(f"   • Плательщик: {account['payer_name']}")
    print(f"   • Адрес:      {account['address']}")
    print("-" * 75)
    print("🏠 Характеристики помещения:")
    print(f"   • Площадь общая:   {account['total_area']} кв.м")
    print(f"   • Площадь жилая:   {account['living_area']} кв.м")
    print(f"   • Зарегистрировано: {account['registered_count']} чел. | Проживает: {account['residents_count']} чел.")
    print("-" * 75)
    print("🏢 Характеристики многоквартирного дома (МКД):")
    print(f"   • Общая площадь дома:       {account['house_total_area']:,.2f} кв.м")
    print(f"   • Жилые помещения дома:     {account['house_living_area']:,.2f} кв.м")
    print(f"   • Места общего пользования: {account['house_common_area']:,.2f} кв.м")
    print(f"   • Доля квартиры в МКД:      {account['apartment_share']:.4f} %")
    print("=" * 75)
    print()

# Список загруженных квитанций с расчетом удельной стоимости 1 кв.м
query_receipts = """
    SELECT 
        id,
        period AS 'Период',
        period_date,
        total_to_pay AS 'К оплате, руб.',
        total_area AS 'Площадь, кв.м',
        ROUND(total_to_pay / total_area, 2) AS 'Руб./кв.м',
        apartment_share AS 'Доля в МКД, %',
        source_file AS 'Файл'
    FROM receipts 
    ORDER BY period_date;
"""

if HAS_PANDAS:
    df_receipts = pd.read_sql_query(query_receipts, conn)
    display(df_receipts)
else:
    rows = cursor.execute(query_receipts).fetchall()
    for r in rows:
        print(dict(r))


👤 Лицевой счет: 32546-580
   • Плательщик: БЕГМА МАКСИМ ВАЛЕНТИНОВИЧ
   • Адрес:      141102, МОСКОВСКАЯ ОБЛ., Г.О. ЩЁЛКОВО, Г ЩЁЛКОВО, УЛ ШМИДТА, д.1, кв.244
---------------------------------------------------------------------------
🏠 Характеристики помещения:
   • Площадь общая:   68.1 кв.м
   • Площадь жилая:   0.0 кв.м
   • Зарегистрировано: 1 чел. | Проживает: 1 чел.
---------------------------------------------------------------------------
🏢 Характеристики многоквартирного дома (МКД):
   • Общая площадь дома:       18,691.20 кв.м
   • Жилые помещения дома:     17,357.80 кв.м
   • Места общего пользования: 2,593.00 кв.м
   • Доля квартиры в МКД:      0.3923 %



,id,Период,period_date,"К оплате, руб.","Площадь, кв.м",Руб./кв.м,"Доля в МКД, %",Файл
0,14,январь 2025,2025-01-01,15593.17,68.1,228.97,0.3923,01 2025.pdf
1,15,февраль 2025,2025-02-01,15630.20,68.1,229.52,0.3923,02 2025.pdf
2,16,март 2025,2025-03-01,14980.09,68.1,219.97,0.3923,03 2025.pdf
3,17,апрель 2025,2025-04-01,14112.21,68.1,207.23,0.3923,04 2025.pdf
4,18,май 2025,2025-05-01,12715.53,68.1,186.72,0.3923,05 2025.pdf
5,19,июнь 2025,2025-06-01,9841.33,68.1,144.51,0.3923,06 2025.pdf
6,20,июль 2025,2025-07-01,9558.61,68.1,140.36,0.3923,07 2025.pdf
7,21,август 2025,2025-08-01,9833.63,68.1,144.40,0.3923,08 2025.pdf
8,10,сентябрь 2025,2025-09-01,11698.69,68.1,171.79,0.3923,09 2025.pdf
9,11,октябрь 2025,2025-10-01,16411.00,68.1,240.98,0.3923,10 2025.pdf


---
## 2. Динамика общей суммы начислений и стоимости 1 кв.м по месяцам
График общей суммы к оплате (без учета добровольного страхования) и расчет удельной стоимости жилищно-коммунальных услуг в расчете на 1 кв.м общей площади помещения.


In [81]:
query_totals = """
    SELECT 
        period_date,
        period,
        total_to_pay,
        total_area,
        ROUND(total_to_pay / total_area, 2) AS cost_per_sqm
    FROM receipts
    ORDER BY period_date;
"""

if HAS_PLOTLY and HAS_PANDAS:
    df_totals = pd.read_sql_query(query_totals, conn)
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=df_totals["period"],
        y=df_totals["total_to_pay"],
        marker_color="#2b5c8f",
        marker_line=dict(color="#1a365d", width=1),
        text=[f"{v:,.0f} ₽<br>({sqm:.1f} ₽/м²)" for v, sqm in zip(df_totals["total_to_pay"], df_totals["cost_per_sqm"])],
        textposition="outside",
        customdata=df_totals[["cost_per_sqm", "total_area"]].values,
        hovertemplate="<b>Период:</b> %{x}<br><b>К оплате:</b> %{y:,.2f} ₽<br><b>Удельная стоимость:</b> %{customdata[0]:,.2f} ₽/м²<br><b>Площадь:</b> %{customdata[1]:.2f} м²<extra></extra>",
        name="К оплате"
    ))
    fig.update_layout(
        title="<b>Динамика суммы к оплате и удельной стоимости 1 кв.м</b>",
        xaxis_title="Период",
        yaxis_title="Сумма к оплате, руб.",
        yaxis_ticksuffix=" ₽",
        height=500,
        margin=dict(t=60, b=60, l=60, r=40)
    )
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    df_totals = pd.read_sql_query(query_totals, conn)
    fig, ax = plt.subplots(figsize=(10, 4.5))
    x_pos = range(len(df_totals))
    bars = ax.bar(x_pos, df_totals["total_to_pay"], color="#2b5c8f", width=0.55, edgecolor="black", alpha=0.85)
    ax.set_title("Динамика суммы к оплате по месяцам (руб.)", fontsize=14, pad=12)
    ax.set_ylabel("Сумма к оплате, руб.")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(df_totals["period"], rotation=25)
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    for bar, sqm in zip(bars, df_totals["cost_per_sqm"]):
        yval = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, yval + 250, f"{yval:,.0f} ₽\n({sqm:.1f} ₽/м²)", ha="center", va="bottom", fontsize=8.5)
    ax.set_ylim(0, df_totals["total_to_pay"].max() * 1.18)
    plt.tight_layout()
    plt.show()
else:
    for r in conn.execute(query_totals):
        print(f"{r['period']}: {r['total_to_pay']} руб. ({r['cost_per_sqm']} руб/кв.м)")


---
## 3. Распределение начислений по категориям услуг
Сравнение долей жилищных, коммунальных и иных услуг в общей сумме начислений.

In [82]:
query_categories = """
    SELECT service_category, ROUND(SUM(total_amount), 2) AS category_sum
    FROM service_charges
    GROUP BY service_category
    ORDER BY category_sum DESC;
"""

if HAS_PLOTLY and HAS_PANDAS:
    df_cats = pd.read_sql_query(query_categories, conn)
    colors = ["#3498db", "#e67e22", "#2ecc71"]
    fig = go.Figure(data=[go.Pie(
        labels=df_cats["service_category"],
        values=df_cats["category_sum"],
        hole=0.4,
        pull=[0.05, 0, 0],
        marker=dict(colors=colors, line=dict(color="#ffffff", width=2)),
        textinfo="label+percent",
        hovertemplate="<b>%{label}</b><br>Сумма: %{value:,.2f} ₽<br>Доля: %{percent}<extra></extra>"
    )])
    fig.update_layout(
        title="<b>Структура затрат по категориям за все периоды</b>",
        legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5),
        height=500
    )
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    df_cats = pd.read_sql_query(query_categories, conn)
    fig, ax = plt.subplots(figsize=(7, 7))
    colors = ["#3498db", "#e67e22", "#2ecc71"]
    wedges, texts, autotexts = ax.pie(
        df_cats["category_sum"],
        labels=df_cats["service_category"],
        autopct="%1.1f%%",
        startangle=0,
        colors=colors,
        explode=[0.03, 0.03, 0.08],
        shadow=False
    )
    for at in autotexts:
        at.set_fontsize(11)
        at.set_weight("bold")
    ax.set_title("Структура затрат по категориям за все периоды", fontsize=13, pad=15)
    plt.show()
else:
    for r in conn.execute(query_categories):
        print(dict(r))


---
## 4. Сезонный анализ отопления
Динамика расхода (Гкал) и суммы начислений за отопление по месяцам (демонстрирует окончание отопительного периода весной).

In [83]:
query_heating = """
    SELECT r.period, r.period_date, sc.volume, sc.tariff, sc.total_amount
    FROM service_charges sc
    JOIN receipts r ON sc.receipt_id = r.id
    WHERE sc.service_name LIKE '%ОТОПЛЕНИЕ%'
    ORDER BY r.period_date;
"""

if HAS_PLOTLY and HAS_PANDAS:
    df_heating = pd.read_sql_query(query_heating, conn)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Bar(
            x=df_heating["period"],
            y=df_heating["total_amount"],
            name="Начислено, ₽",
            marker_color="#e74c3c",
            opacity=0.8,
            hovertemplate="<b>%{x}</b><br>Начислено: %{y:,.2f} ₽<extra></extra>"
        ),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(
            x=df_heating["period"],
            y=df_heating["volume"],
            name="Объем, Гкал",
            mode="lines+markers",
            line=dict(color="#2c3e50", width=3),
            marker=dict(size=8, symbol="circle"),
            hovertemplate="<b>%{x}</b><br>Объем тепла: %{y:.4f} Гкал<extra></extra>"
        ),
        secondary_y=True
    )
    fig.update_layout(
        title="<b>Отопление: объем потребления (Гкал) и сумма начисления (руб.)</b>",
        xaxis_title="Период",
        hovermode="x unified",
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
        height=480
    )
    fig.update_yaxes(title_text="Сумма начислений, руб.", secondary_y=False, ticksuffix=" ₽")
    fig.update_yaxes(title_text="Объем потребления, Гкал", secondary_y=True, ticksuffix=" Гкал")
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    df_heating = pd.read_sql_query(query_heating, conn)
    fig, ax1 = plt.subplots(figsize=(10, 4.5))
    ax2 = ax1.twinx()
    x_pos = range(len(df_heating))
    bars = ax1.bar(x_pos, df_heating["total_amount"], color="#e74c3c", alpha=0.7, width=0.45, label="Сумма, руб.")
    line = ax2.plot(x_pos, df_heating["volume"], color="#2c3e50", marker="o", linewidth=2.5, label="Объем, Гкал")
    ax1.set_title("Отопление: объем потребления (Гкал) и сумма начисления (руб.)", fontsize=13, pad=12)
    ax1.set_ylabel("Сумма, руб.", color="#c0392b")
    ax2.set_ylabel("Объем, Гкал", color="#2c3e50")
    ax1.set_ylim(0, df_heating["total_amount"].max() * 1.2)
    ax2.set_ylim(0, df_heating["volume"].max() * 1.2)
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(df_heating["period"], rotation=25)
    lines = [bars, line[0]]
    labels = ["Сумма, руб.", "Объем, Гкал"]
    ax1.legend(lines, labels, loc="upper right")
    fig.tight_layout()
    plt.show()
else:
    for r in conn.execute(query_heating):
        print(dict(r))


---
## 5. Динамика расхода электроэнергии и водоотведения
Отслеживание объемов потребления ресурсов в физических величинах (кВт*ч и куб.м).

In [84]:
query_resources = """
    SELECT r.period_date, r.period, sc.service_name, sc.volume, sc.unit
    FROM service_charges sc
    JOIN receipts r ON sc.receipt_id = r.id
    WHERE sc.service_name LIKE '%ЭЛЕКТРИЧЕСТВО%' OR sc.service_name = 'ВОДООТВЕДЕНИЕ'
    ORDER BY r.period_date;
"""

if HAS_PLOTLY and HAS_PANDAS:
    df_res = pd.read_sql_query(query_resources, conn)
    pivot_res = df_res.pivot(index="period_date", columns="service_name", values="volume").sort_index()
    period_map = df_res.drop_duplicates("period_date").set_index("period_date")["period"]
    x_labels = [period_map.get(d, d) for d in pivot_res.index]
    
    water_cols = [c for c in pivot_res.columns if "ВОДООТВЕДЕНИЕ" in c]
    elec_cols = [c for c in pivot_res.columns if "ЭЛЕКТРИЧЕСТВО" in c]
    water_col = water_cols[0] if water_cols else pivot_res.columns[0]
    elec_col = elec_cols[0] if elec_cols else pivot_res.columns[-1]
    
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(
        go.Scatter(
            x=x_labels,
            y=pivot_res[water_col],
            name="Водоотведение (м³)",
            mode="lines+markers",
            line=dict(color="#2980b9", width=2.5),
            marker=dict(size=8, symbol="square"),
            hovertemplate="<b>%{x}</b><br>Водоотведение: <b>%{y:.2f} м³</b><extra></extra>"
        ),
        secondary_y=False
    )
    fig.add_trace(
        go.Scatter(
            x=x_labels,
            y=pivot_res[elec_col],
            name="Электроэнергия (кВт⋅ч)",
            mode="lines+markers",
            line=dict(color="#e67e22", width=2.5),
            marker=dict(size=8, symbol="triangle-up"),
            hovertemplate="<b>%{x}</b><br>Электроэнергия: <b>%{y:.2f} кВт⋅ч</b><extra></extra>"
        ),
        secondary_y=True
    )
    fig.update_layout(
        title="<b>Динамика расхода электроэнергии и водоотведения</b>",
        xaxis_title="Период",
        hovermode="x unified",
        legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
        height=480
    )
    fig.update_yaxes(title_text="Водоотведение, куб.м", secondary_y=False, ticksuffix=" м³")
    fig.update_yaxes(title_text="Электроэнергия, кВт⋅ч", secondary_y=True, ticksuffix=" кВт⋅ч")
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    df_res = pd.read_sql_query(query_resources, conn)
    pivot_res = df_res.pivot(index="period_date", columns="service_name", values="volume").sort_index()
    period_map = df_res.drop_duplicates("period_date").set_index("period_date")["period"]
    x_labels = [period_map.get(d, d) for d in pivot_res.index]
    x_indices = range(len(pivot_res))
    water_cols = [c for c in pivot_res.columns if "ВОДООТВЕДЕНИЕ" in c]
    elec_cols = [c for c in pivot_res.columns if "ЭЛЕКТРИЧЕСТВО" in c]
    water_col = water_cols[0] if water_cols else pivot_res.columns[0]
    elec_col = elec_cols[0] if elec_cols else pivot_res.columns[-1]
    fig, ax1 = plt.subplots(figsize=(10, 4.5))
    ax2 = ax1.twinx()
    line1 = ax1.plot(x_indices, pivot_res[water_col], marker="o", color="#2980b9", linewidth=2.2, label="Водоотведение (куб.м)")
    line2 = ax2.plot(x_indices, pivot_res[elec_col], marker="s", color="#e67e22", linewidth=2.2, label="Электричество (кВт*ч)")
    ax1.set_title("Динамика потребления ресурсов", fontsize=13, pad=12)
    ax1.set_xlabel("Период")
    ax1.set_ylabel("Водоотведение, куб.м", color="#2980b9", fontsize=11)
    ax2.set_ylabel("Электричество, кВт*ч", color="#d35400", fontsize=11)
    ax1.set_xticks(x_indices)
    ax1.set_xticklabels(x_labels, rotation=30)
    ax1.grid(True, linestyle="--", alpha=0.6)
    lines = line1 + line2
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, loc="upper left")
    fig.tight_layout()
    plt.show()
else:
    for r in conn.execute(query_resources):
        print(dict(r))


---
## 6. Топ-10 самых затратных услуг за весь период

In [85]:
query_top = """
    SELECT service_name, ROUND(SUM(total_amount), 2) AS total_sum
    FROM service_charges
    GROUP BY service_name
    ORDER BY total_sum DESC
    LIMIT 10;
"""

if HAS_PLOTLY and HAS_PANDAS:
    df_top = pd.read_sql_query(query_top, conn).sort_values("total_sum", ascending=True)
    fig = go.Figure(go.Bar(
        x=df_top["total_sum"],
        y=df_top["service_name"],
        orientation="h",
        marker_color="#16a085",
        marker_line=dict(color="#0e6655", width=1),
        text=[f"{v:,.2f} ₽" for v in df_top["total_sum"]],
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Суммарные начисления: <b>%{x:,.2f} ₽</b><extra></extra>"
    ))
    fig.update_layout(
        title="<b>Топ-10 самых затратных услуг за весь наблюдаемый период</b>",
        xaxis_title="Суммарные начисления, руб.",
        xaxis_ticksuffix=" ₽",
        height=520,
        margin=dict(l=230, r=70, t=60, b=60)
    )
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    df_top = pd.read_sql_query(query_top, conn)
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(df_top["service_name"][::-1], df_top["total_sum"][::-1], color="#16a085", height=0.6)
    ax.set_title("Топ-10 затратных услуг за весь наблюдаемый период", fontsize=13, pad=12)
    ax.set_xlabel("Суммарные начисления, руб.")
    for bar in bars:
        w = bar.get_width()
        ax.text(w + 300, bar.get_y() + bar.get_height() / 2, f"{w:,.2f} ₽", va="center", fontsize=9)
    ax.set_xlim(0, df_top["total_sum"].max() * 1.15)
    plt.tight_layout()
    plt.show()
else:
    for r in conn.execute(query_top):
        print(f"{r['service_name']}: {r['total_sum']} руб.")


---
## 7. Сведения о перерасчетах
Выборка всех услуг, по которым производились перерасчеты (`recalculation != 0`).

In [86]:
query_recalc = """
    SELECT r.period, sc.service_name, sc.recalculation, sc.total_amount
    FROM service_charges sc
    JOIN receipts r ON sc.receipt_id = r.id
    WHERE ABS(sc.recalculation) > 0.01
    ORDER BY r.period_date, sc.service_name;
"""

if HAS_PLOT_LIBS:
    df_recalc = pd.read_sql_query(query_recalc, conn)
    if not df_recalc.empty:
        display(df_recalc)
    else:
        print("Перерасчетов за данный период не обнаружено.")
else:
    rows = conn.execute(query_recalc).fetchall()
    for r in rows:
        print(dict(r))

,period,service_name,recalculation,total_amount
0,май 2025,ВОДООТВЕДЕНИЕ ОДН,194.82,199.54
1,май 2025,ХОЛОДНОЕ В/С ОДН,217.78,220.34
2,май 2025,ЭЛЕКТРОСНАБЖЕНИЕ ОДН,66.08,236.04
3,июнь 2025,ВОДООТВЕДЕНИЕ ОДН,194.82,199.54
4,июнь 2025,ХОЛОДНОЕ В/С ОДН,217.78,220.34
5,июнь 2025,ЭЛЕКТРОСНАБЖЕНИЕ ОДН,66.08,236.04
6,июль 2025,ВОДООТВЕДЕНИЕ ОДН,194.82,200.56
7,июль 2025,ХОЛОДНОЕ В/С ОДН,217.78,220.39
8,июль 2025,ЭЛЕКТРОСНАБЖЕНИЕ ОДН,66.08,267.47
9,август 2025,ВОДООТВЕДЕНИЕ ОДН,194.82,200.56


---
## 8. Топ-3 самых затратных услуг по годам
Сравнение трех ключевых статей расходов в квитанциях за каждый год с расчетом суммы начислений и их процентной доли в годовом бюджете ЖКХ.


In [87]:
query_top3_yearly = """
WITH year_totals AS (
    SELECT year, SUM(total_amount) AS year_sum
    FROM service_charges sc
    JOIN receipts r ON sc.receipt_id = r.id
    GROUP BY year
),
ranked_services AS (
    SELECT 
        r.year,
        sc.service_name,
        ROUND(SUM(sc.total_amount), 2) AS total_sum,
        ROUND(SUM(sc.total_amount) * 100.0 / yt.year_sum, 1) AS share_percent,
        ROW_NUMBER() OVER (PARTITION BY r.year ORDER BY SUM(sc.total_amount) DESC) AS rank
    FROM service_charges sc
    JOIN receipts r ON sc.receipt_id = r.id
    JOIN year_totals yt ON r.year = yt.year
    GROUP BY r.year, sc.service_name
)
SELECT year AS 'Год', rank AS 'Место', service_name AS 'Услуга', total_sum AS 'Сумма, руб.', share_percent AS 'Доля, %'
FROM ranked_services
WHERE rank <= 3
ORDER BY year, rank;
"""

if HAS_PLOTLY and HAS_PANDAS:
    df_top3 = pd.read_sql_query(query_top3_yearly, conn)
    display(df_top3)
    years = sorted(df_top3['Год'].unique())
    colors_rank = {1: '#c0392b', 2: '#2980b9', 3: '#16a085'}
    
    fig = make_subplots(
        rows=1, cols=len(years),
        subplot_titles=[f"<b>Топ-3 услуг за {y} год</b>" for y in years],
        shared_yaxes=False,
        horizontal_spacing=0.18
    )
    for col_idx, year in enumerate(years, start=1):
        df_y = df_top3[df_top3['Год'] == year].sort_values("Сумма, руб.", ascending=True)
        fig.add_trace(
            go.Bar(
                x=df_y["Сумма, руб."],
                y=df_y["Услуга"],
                orientation="h",
                marker_color=[colors_rank.get(m, '#3498db') for m in df_y["Место"]],
                text=[f"{s:,.0f} ₽ ({d}%)" for s, d in zip(df_y["Сумма, руб."], df_y["Доля, %"])],
                textposition="auto",
                customdata=df_y[["Место", "Доля, %", "Год"]].values,
                hovertemplate="<b>%{y}</b><br>Год: %{customdata[2]}<br>Место: #%{customdata[0]}<br>Сумма: <b>%{x:,.2f} ₽</b><br>Доля в бюджете года: <b>%{customdata[1]}%</b><extra></extra>",
                showlegend=False
            ),
            row=1, col=col_idx
        )
        fig.update_xaxes(title_text="Сумма начислений, руб.", ticksuffix=" ₽", row=1, col=col_idx)
    fig.update_layout(
        title="<b>Топ-3 самых затратных услуг в разрезе лет</b>",
        height=450,
        margin=dict(t=80, b=50, l=190, r=40)
    )
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    df_top3 = pd.read_sql_query(query_top3_yearly, conn)
    display(df_top3)
    years = sorted(df_top3['Год'].unique())
    fig, axes = plt.subplots(1, len(years), figsize=(6.5 * len(years), 4.5), sharey=False)
    if len(years) == 1:
        axes = [axes]
    colors = ['#c0392b', '#2980b9', '#16a085']
    for idx, year in enumerate(years):
        ax = axes[idx]
        df_y = df_top3[df_top3['Год'] == year].copy().iloc[::-1]
        y_pos = range(len(df_y))
        bars = ax.barh(y_pos, df_y['Сумма, руб.'], color=colors, height=0.55, edgecolor='black', alpha=0.85)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(df_y['Услуга'], fontsize=10)
        ax.set_title(f"Топ-3 услуг за {year} год", fontsize=13, pad=12)
        ax.set_xlabel("Сумма начислений, руб.")
        ax.grid(axis='x', linestyle='--', alpha=0.6)
        max_val = df_y['Сумма, руб.'].max()
        ax.set_xlim(0, max_val * 1.35)
        for bar, (_, row) in zip(bars, df_y.iterrows()):
            w = bar.get_width()
            label_text = f"{w:,.2f} ₽ ({row['Доля, %']} %)"
            ax.text(w + max_val * 0.02, bar.get_y() + bar.get_height() / 2, label_text, va='center', fontsize=9.5, fontweight='bold')
    fig.suptitle("Топ-3 самых затратных услуг в разрезе лет", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    rows = conn.execute(query_top3_yearly).fetchall()
    for r in rows:
        print(dict(r))


,Год,Место,Услуга,"Сумма, руб.","Доля, %"
0,2025,1,ОТОПЛЕНИЕ,38625.38,23.3
1,2025,2,СОДЕРЖАНИЕ Ж/Ф,33817.78,20.4
2,2025,3,ГОРЯЧЕЕ В/С (ЭНЕРГИЯ),27918.43,16.9
3,2026,1,ОТОПЛЕНИЕ,30774.11,22.6
4,2026,2,СОДЕРЖАНИЕ Ж/Ф,27596.82,20.2
5,2026,3,ГОРЯЧЕЕ В/С (ЭНЕРГИЯ),23725.16,17.4


---
## 9. Распределение начислений по категориям услуг в разрезе годов
Сравнение начислений по жилищным, коммунальным и иным услугам для каждого года: абсолютные суммы в рублях и соотношение процентных долей.


In [88]:
query_cat_yearly = """
WITH year_totals AS (
    SELECT year, SUM(total_amount) AS total_year
    FROM service_charges sc
    JOIN receipts r ON sc.receipt_id = r.id
    GROUP BY year
)
SELECT 
    r.year AS 'Год',
    sc.service_category AS 'Категория',
    ROUND(SUM(sc.total_amount), 2) AS 'Сумма, руб.',
    ROUND(SUM(sc.total_amount) * 100.0 / yt.total_year, 1) AS 'Доля, %'
FROM service_charges sc
JOIN receipts r ON sc.receipt_id = r.id
JOIN year_totals yt ON r.year = yt.year
GROUP BY r.year, sc.service_category
ORDER BY r.year, SUM(sc.total_amount) DESC;
"""

if HAS_PLOTLY and HAS_PANDAS:
    df_cat_year = pd.read_sql_query(query_cat_yearly, conn)
    display(df_cat_year)
    
    pivot_sum = df_cat_year.pivot(index='Год', columns='Категория', values='Сумма, руб.')
    pivot_share = df_cat_year.pivot(index='Год', columns='Категория', values='Доля, %')
    cat_order = [c for c in ['Коммунальные услуги', 'Жилищные услуги', 'Иные услуги'] if c in pivot_sum.columns]
    palette = {'Коммунальные услуги': '#3498db', 'Жилищные услуги': '#e67e22', 'Иные услуги': '#2ecc71'}
    years_str = [str(y) for y in pivot_sum.index]
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["<b>Суммы начислений по категориям (руб.)</b>", "<b>Соотношение долей категорий (%)</b>"],
        horizontal_spacing=0.14
    )
    # 1. Grouped bar
    for cat in cat_order:
        fig.add_trace(
            go.Bar(
                x=years_str,
                y=pivot_sum[cat],
                name=cat,
                marker_color=palette.get(cat, '#95a5a6'),
                text=[f"{v:,.0f} ₽" for v in pivot_sum[cat]],
                textposition="outside",
                hovertemplate="<b>%{x} год</b><br>Категория: " + cat + "<br>Сумма: <b>%{y:,.2f} ₽</b><extra></extra>",
                offsetgroup=cat
            ),
            row=1, col=1
        )
    # 2. 100% Stacked bar
    for cat in cat_order:
        fig.add_trace(
            go.Bar(
                x=years_str,
                y=pivot_share[cat],
                name=cat,
                marker_color=palette.get(cat, '#95a5a6'),
                text=[f"{v:.1f}%" for v in pivot_share[cat]],
                textposition="inside",
                hovertemplate="<b>%{x} год</b><br>Категория: " + cat + "<br>Доля: <b>%{y:.1f}%</b><extra></extra>",
                offsetgroup="stacked",
                showlegend=False
            ),
            row=1, col=2
        )
    fig.update_layout(
        title="<b>Распределение начислений по категориям услуг в разрезе годов</b>",
        legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
        height=500
    )
    fig.update_xaxes(title_text="Год", row=1, col=1)
    fig.update_xaxes(title_text="Год", row=1, col=2)
    fig.update_yaxes(title_text="Сумма начислений, руб.", ticksuffix=" ₽", row=1, col=1)
    fig.update_yaxes(title_text="Доля, %", ticksuffix="%", range=[0, 105], row=1, col=2)
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    df_cat_year = pd.read_sql_query(query_cat_yearly, conn)
    display(df_cat_year)
    pivot_sum = df_cat_year.pivot(index='Год', columns='Категория', values='Сумма, руб.')
    pivot_share = df_cat_year.pivot(index='Год', columns='Категория', values='Доля, %')
    cat_order = [c for c in ['Коммунальные услуги', 'Жилищные услуги', 'Иные услуги'] if c in pivot_sum.columns]
    pivot_sum = pivot_sum[cat_order]
    pivot_share = pivot_share[cat_order]
    colors = ['#3498db', '#e67e22', '#2ecc71']
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    pivot_sum.plot(kind='bar', ax=ax1, color=colors, width=0.6, edgecolor='black', alpha=0.85)
    ax1.set_title("Суммы начислений по категориям за каждый год", fontsize=13, pad=12)
    ax1.set_ylabel("Сумма начислений, руб.")
    ax1.set_xlabel("Год")
    ax1.set_xticklabels(pivot_sum.index, rotation=0)
    ax1.grid(axis='y', linestyle='--', alpha=0.6)
    ax1.legend(title="Категория", loc="upper right")
    for p in ax1.patches:
        height = p.get_height()
        if height > 0:
            ax1.annotate(f"{height:,.0f} ₽", (p.get_x() + p.get_width() / 2., height), ha='center', va='bottom', fontsize=8.5, xytext=(0, 2), textcoords='offset points')
    ax1.set_ylim(0, pivot_sum.values.max() * 1.15)
    pivot_share.plot(kind='bar', stacked=True, ax=ax2, color=colors, width=0.45, edgecolor='black', alpha=0.85)
    ax2.set_title("Соотношение долей категорий в %", fontsize=13, pad=12)
    ax2.set_ylabel("Доля, %")
    ax2.set_xlabel("Год")
    ax2.set_xticklabels(pivot_share.index, rotation=0)
    ax2.set_ylim(0, 105)
    ax2.grid(axis='y', linestyle='--', alpha=0.6)
    ax2.legend(title="Категория", loc="lower right")
    for n, year in enumerate(pivot_share.index):
        cum_height = 0
        for col in cat_order:
            val = pivot_share.loc[year, col]
            if val > 3.0:
                ax2.text(n, cum_height + val / 2, f"{val:.1f}%", ha='center', va='center', fontsize=10, fontweight='bold', color='white')
            cum_height += val
    fig.suptitle("Распределение начислений по категориям услуг в разрезе годов", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    for r in conn.execute(query_cat_yearly):
        print(dict(r))


,Год,Категория,"Сумма, руб.","Доля, %"
0,2025,Коммунальные услуги,108412.45,65.5
1,2025,Жилищные услуги,56504.90,34.1
2,2025,Иные услуги,630.00,0.4
3,2026,Коммунальные услуги,88816.90,65.1
4,2026,Жилищные услуги,47044.62,34.5
5,2026,Иные услуги,555.00,0.4


---
## 10. Годовая динамика и сравнение год к году (Year-over-Year / YoY)
Наглядное сопоставление платежей между годами:
- **Сезонные кривые месяц к месяцу** (наложение линий 2025 vs 2026 года для одинаковых месяцев 1–12).
- **Диаграмма прироста** (дельта в рублях и процентах по каждому месяцу).
- **Среднемесячный чек** и средний индекс изменения расходов.


In [89]:
query_yoy = """
    SELECT month, year, total_to_pay
    FROM receipts
    ORDER BY month, year;
"""

if HAS_PANDAS:
    df_raw = pd.read_sql_query(query_yoy, conn)
    pivot_yoy = df_raw.pivot(index='month', columns='year', values='total_to_pay').sort_index()
    month_names = ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек']
    pivot_yoy.index = [month_names[m - 1] for m in pivot_yoy.index]
    years = sorted(pivot_yoy.columns)
    table_df = pivot_yoy.copy()
    
    if len(years) >= 2:
        prev_y, curr_y = years[-2], years[-1]
        diff_rub = table_df[curr_y] - table_df[prev_y]
        diff_pct = (diff_rub / table_df[prev_y]) * 100.0
        table_df[f"Прирост {curr_y} vs {prev_y}, руб."] = diff_rub.round(2)
        table_df["Прирост, %"] = diff_pct.round(1)
        
        avg_prev = table_df[prev_y].dropna().mean()
        avg_curr = table_df[curr_y].dropna().mean()
        avg_diff_pct = ((avg_curr - avg_prev) / avg_prev) * 100.0
        print("=" * 65)
        print(f"📊 Сравнительный анализ платежей {prev_y} vs {curr_y} гг.:")
        print(f"   • Среднемесячный платеж в {prev_y} г.:  {avg_prev:,.2f} ₽")
        print(f"   • Среднемесячный платеж в {curr_y} г.:  {avg_curr:,.2f} ₽")
        print(f"   • Среднее изменение платежа (YoY):  {avg_diff_pct:+.1f} % ({avg_curr - avg_prev:+,.2f} ₽)")
        print("=" * 65)
    display(table_df)

if HAS_PLOTLY and HAS_PANDAS:
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=[
            "<b>Сезонные кривые начислений по месяцам (Year-over-Year)</b>",
            f"<b>Разница в начислениях: {curr_y} г. относительно {prev_y} г. (руб.)</b>" if len(years) >= 2 else "<b>Разница начислений</b>"
        ],
        vertical_spacing=0.15
    )
    colors_palette = ['#2980b9', '#c0392b', '#27ae60', '#8e44ad']
    for idx, y in enumerate(years):
        fig.add_trace(
            go.Scatter(
                x=pivot_yoy.index,
                y=pivot_yoy[y],
                name=f"{y} год",
                mode="lines+markers",
                line=dict(color=colors_palette[idx % len(colors_palette)], width=2.5),
                marker=dict(size=7),
                hovertemplate=f"<b>%{{x}} ({y} г.)</b><br>Начислено: <b>%{{y:,.2f}} ₽</b><extra></extra>"
            ),
            row=1, col=1
        )
    if len(years) >= 2:
        diff_col_name = f"Прирост {curr_y} vs {prev_y}, руб."
        valid_diff = table_df[[diff_col_name, 'Прирост, %']].dropna()
        diff_vals = valid_diff[diff_col_name]
        diff_pcts = valid_diff['Прирост, %']
        bar_colors = ['#e74c3c' if v >= 0 else '#27ae60' for v in diff_vals]
        
        fig.add_trace(
            go.Bar(
                x=valid_diff.index,
                y=diff_vals,
                marker_color=bar_colors,
                text=[f"{v:+,.0f} ₽ ({p:+.1f}%)" for v, p in zip(diff_vals, diff_pcts)],
                textposition="auto",
                customdata=diff_pcts,
                hovertemplate="<b>%{x}</b><br>Прирост: <b>%{y:+,.2f} ₽</b><br>Изменение: <b>%{customdata:+.1f}%</b><extra></extra>",
                name="Дельта (руб.)",
                showlegend=False
            ),
            row=2, col=1
        )
    fig.update_layout(
        title="<b>Годовой сравнительный анализ начислений (YoY)</b>",
        hovermode="x unified",
        height=750,
        margin=dict(t=80, b=50, l=60, r=40)
    )
    fig.update_yaxes(title_text="Сумма, руб.", ticksuffix=" ₽", row=1, col=1)
    fig.update_yaxes(title_text="Прирост, руб.", ticksuffix=" ₽", row=2, col=1)
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9))
    colors_palette = ['#2980b9', '#c0392b', '#27ae60', '#8e44ad']
    markers = ['o', 's', '^', 'D']
    x_pos = range(len(pivot_yoy))
    for idx, y in enumerate(years):
        color = colors_palette[idx % len(colors_palette)]
        marker = markers[idx % len(markers)]
        ax1.plot(x_pos, pivot_yoy[y], marker=marker, linewidth=2.5, markersize=6, color=color, label=f"{y} год")
    if len(years) >= 2:
        prev_y, curr_y = years[-2], years[-1]
        valid_mask = pivot_yoy[prev_y].notna() & pivot_yoy[curr_y].notna()
        ax1.fill_between(x_pos, pivot_yoy[prev_y], pivot_yoy[curr_y], where=valid_mask, color='#e74c3c', alpha=0.15, label=f"Разница ({curr_y} vs {prev_y})")
    ax1.set_title("Сезонные кривые начислений по месяцам (Year-over-Year)", fontsize=13, pad=12)
    ax1.set_ylabel("Сумма к оплате, руб.")
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels(pivot_yoy.index, fontsize=10)
    ax1.grid(True, linestyle="--", alpha=0.6)
    ax1.legend(loc="upper right", frameon=True)
    if len(years) >= 2:
        diff_col_name = f"Прирост {curr_y} vs {prev_y}, руб."
        valid_diff = table_df[[diff_col_name]].dropna()
        diff_vals = valid_diff.iloc[:, 0]
        bar_colors = ['#e74c3c' if v >= 0 else '#27ae60' for v in diff_vals]
        diff_x = range(len(valid_diff))
        bars = ax2.bar(diff_x, diff_vals, color=bar_colors, width=0.55, edgecolor='black', alpha=0.85)
        ax2.axhline(0, color='black', linewidth=1, linestyle='-')
        ax2.set_title(f"Разница в начислениях: {curr_y} г. относительно {prev_y} г. (руб.)", fontsize=13, pad=12)
        ax2.set_ylabel("Прирост, руб.")
        ax2.set_xticks(diff_x)
        ax2.set_xticklabels(valid_diff.index, fontsize=10)
        ax2.grid(axis='y', linestyle="--", alpha=0.6)
        for bar, m_idx in zip(bars, valid_diff.index):
            val = bar.get_height()
            pct = table_df.loc[m_idx, 'Прирост, %']
            sign = '+' if val > 0 else ''
            ax2.annotate(f"{sign}{val:,.0f} ₽\n({sign}{pct:.1f}%)", (bar.get_x() + bar.get_width() / 2., val), ha='center', va='bottom' if val >= 0 else 'top', fontsize=8.5, fontweight='bold', xytext=(0, 2 if val >= 0 else -14), textcoords='offset points')
        y_max = max(abs(diff_vals.min()), abs(diff_vals.max())) * 1.35
        ax2.set_ylim(-y_max * 0.35, y_max)
    fig.suptitle("Годовой сравнительный анализ начислений (YoY)", fontsize=15, y=0.99)
    plt.tight_layout()
    plt.show()
else:
    for r in conn.execute(query_yoy):
        print(dict(r))


📊 Сравнительный анализ платежей 2025 vs 2026 гг.:
   • Среднемесячный платеж в 2025 г.:  13,795.61 ₽
   • Среднемесячный платеж в 2026 г.:  15,157.39 ₽
   • Среднее изменение платежа (YoY):  +9.9 % (+1,361.78 ₽)


year,2025,2026,"Прирост 2026 vs 2025, руб.","Прирост, %"
Янв,15593.17,19558.66,3965.49,25.4
Фев,15630.20,20584.76,4954.56,31.7
Мар,14980.09,19493.98,4513.89,30.1
Апр,14112.21,17536.48,3424.27,24.3
Май,12715.53,14781.24,2065.71,16.2
Июн,9841.33,12409.63,2568.30,26.1
Июл,9558.61,10064.11,505.50,5.3
Авг,9833.63,11073.13,1239.50,12.6
Сен,11698.69,10914.53,-784.16,-6.7
Окт,16411.00,NaN,NaN,NaN


---
## 11. Анализ индексации тарифов (Tariff Tracker)
Отслеживание изменений ставок тарифов по ключевым статьям ЖКУ.
График **Индекса тарифов (базовый период = 100%)** демонстрирует чистую инфляцию цен поставщиков коммунальных и жилищных услуг независимо от объемов потребления.


In [90]:
query_tariffs = """
    SELECT 
        r.period_date,
        r.period,
        sc.service_name,
        sc.unit,
        sc.tariff
    FROM service_charges sc
    JOIN receipts r ON sc.receipt_id = r.id
    WHERE sc.tariff > 0
    ORDER BY r.period_date;
"""

if HAS_PANDAS:
    df_t = pd.read_sql_query(query_tariffs, conn)
    pivot_tariffs = df_t.pivot_table(index='period_date', columns='service_name', values='tariff', aggfunc='mean').sort_index()
    period_map = df_t.drop_duplicates('period_date').set_index('period_date')['period']
    x_labels = [period_map.get(d, d) for d in pivot_tariffs.index]
    
    # Выбираем ключевые репрезентативные услуги
    key_services = []
    for pattern in ['ВЗНОС НА К/РЕМ', 'СОДЕРЖАНИЕ Ж/Ф', 'ЭЛЕКТРИЧЕСТВО', 'ОТОПЛЕНИЕ', 'ВОДООТВЕДЕНИЕ', 'ДОМОФОН', 'ОБРАЩЕНИЕ С ТКО']:
        matches = [c for c in pivot_tariffs.columns if pattern in c]
        if matches:
            key_services.append(matches[0])
            
    key_services = list(dict.fromkeys(key_services))
    pivot_tariffs = pivot_tariffs[key_services]
    
    # 1. Таблица изменения тарифов
    summary_rows = []
    for col in pivot_tariffs.columns:
        s = pivot_tariffs[col].dropna()
        if not s.empty:
            first_val = s.iloc[0]
            last_val = s.iloc[-1]
            diff_val = last_val - first_val
            diff_pct = (diff_val / first_val) * 100.0 if first_val > 0 else 0.0
            unit_val = df_t[df_t['service_name'] == col]['unit'].iloc[0] or ''
            summary_rows.append({
                'Услуга': col,
                'Ед. изм.': unit_val,
                'Начальный тариф': round(first_val, 4),
                'Текущий тариф': round(last_val, 4),
                'Прирост (руб.)': round(diff_val, 4),
                'Прирост (%)': round(diff_pct, 1)
            })
    df_summary_t = pd.DataFrame(summary_rows).sort_values('Прирост (%)', ascending=False)
    display(df_summary_t)

if HAS_PLOTLY and HAS_PANDAS:
    pivot_idx = (pivot_tariffs / pivot_tariffs.bfill().iloc[0]) * 100.0
    fig = go.Figure()
    for col in pivot_idx.columns:
        short_name = col.replace('ОДНОТАРИФНЫЙ ПУ (Д1)', '').replace('ВЗНОС НА ', '').strip()
        fig.add_trace(
            go.Scatter(
                x=x_labels,
                y=pivot_idx[col],
                name=short_name,
                mode="lines+markers",
                line=dict(width=2.4),
                marker=dict(size=6),
                customdata=pivot_tariffs[col],
                hovertemplate="<b>" + short_name + "</b><br>Период: %{x}<br>Индекс: <b>%{y:.1f}%</b><br>Тариф: <b>%{customdata:,.4f} ₽</b><extra></extra>"
            )
        )
    fig.update_layout(
        title="<b>Индекс роста тарифов ЖКУ (Базовый период = 100%)</b>",
        xaxis_title="Период",
        yaxis_title="Индекс тарифа, %",
        yaxis_ticksuffix="%",
        hovermode="closest",
        height=520,
        legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5)
    )
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    pivot_idx = (pivot_tariffs / pivot_tariffs.bfill().iloc[0]) * 100.0
    fig, ax = plt.subplots(figsize=(12, 5))
    x_pos = range(len(pivot_idx))
    for col in pivot_idx.columns:
        short_name = col.replace('ОДНОТАРИФНЫЙ ПУ (Д1)', '').replace('ВЗНОС НА ', '').strip()
        ax.plot(x_pos, pivot_idx[col], marker='o', linewidth=2.2, label=f"{short_name}")
    ax.set_title("Индекс роста тарифов ЖКУ (Базовый период = 100%)", fontsize=13, pad=12)
    ax.set_ylabel("Индекс тарифа, %")
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels, rotation=30)
    ax.grid(True, linestyle="--", alpha=0.6)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
    plt.tight_layout()
    plt.show()
else:
    for r in conn.execute(query_tariffs):
        print(dict(r))


,Услуга,Ед. изм.,Начальный тариф,Текущий тариф,Прирост (руб.),Прирост (%)
4,ДОМОФОН,рубль,50.0000,65.0000,15.000,30.0
3,ВОДООТВЕДЕНИЕ,куб. м.,41.6000,51.4700,9.870,23.7
0,СОДЕРЖАНИЕ Ж/Ф,кв.м.,36.7900,45.4300,8.640,23.5
1,ЭЛЕКТРИЧЕСТВО ОДНОТАРИФНЫЙ ПУ (Д1),кВт*ч,5.5700,6.7000,1.130,20.3
2,ОТОПЛЕНИЕ,Гкал,3329.9000,3845.4800,515.580,15.5
5,ОБРАЩЕНИЕ С ТКО,кв.м.,9.9859,11.5229,1.537,15.4


---
## 12. Факторный анализ: «Тариф vs Объем» (Price-Volume Effect)
Декомпозиция изменения начислений между сопоставимыми периодами (**Сентябрь 2025 vs Сентябрь 2026**).
Общее изменение начислений раскладывается на две составляющие:
1. **Фактор объема** $(V_1 - V_0) \times T_0$ — изменение за счет расхода ресурса в натуральных величинах.
2. **Фактор тарифа** $V_1 \times (T_1 - T_0)$ — изменение за счет индексации ставок цен поставщиками.


In [91]:
query_factor = """
SELECT 
    sc.service_name,
    MAX(CASE WHEN r.year = 2025 THEN sc.volume END) AS v0,
    MAX(CASE WHEN r.year = 2025 THEN sc.tariff END) AS t0,
    MAX(CASE WHEN r.year = 2025 THEN sc.billed_amount END) AS b0,
    MAX(CASE WHEN r.year = 2026 THEN sc.volume END) AS v1,
    MAX(CASE WHEN r.year = 2026 THEN sc.tariff END) AS t1,
    MAX(CASE WHEN r.year = 2026 THEN sc.billed_amount END) AS b1
FROM service_charges sc
JOIN receipts r ON sc.receipt_id = r.id
WHERE r.month = 9
GROUP BY sc.service_name
HAVING b0 IS NOT NULL AND b1 IS NOT NULL AND (b0 > 0 OR b1 > 0)
ORDER BY b1 DESC;
"""

if HAS_PANDAS:
    df_fact = pd.read_sql_query(query_factor, conn)
    for col in ['v0', 't0', 'b0', 'v1', 't1', 'b1']:
        df_fact[col] = df_fact[col].fillna(0.0)
        
    df_fact['Итого_Дельта_руб'] = (df_fact['b1'] - df_fact['b0']).round(2)
    df_fact['Эффект_Объема_руб'] = ((df_fact['v1'] - df_fact['v0']) * df_fact['t0']).round(2)
    df_fact['Эффект_Тарифа_руб'] = (df_fact['v1'] * (df_fact['t1'] - df_fact['t0'])).round(2)
    
    cols_show = ['service_name', 'b0', 'b1', 'Итого_Дельта_руб', 'Эффект_Объема_руб', 'Эффект_Тарифа_руб']
    df_display_fact = df_fact[cols_show].rename(columns={
        'service_name': 'Услуга',
        'b0': 'Сен 2025, руб.',
        'b1': 'Сен 2026, руб.',
        'Итого_Дельта_руб': 'Общее изменение, руб.',
        'Эффект_Объема_руб': 'Фактор объема, руб.',
        'Эффект_Тарифа_руб': 'Фактор тарифа, руб.'
    })
    display(df_display_fact)

if HAS_PLOTLY and HAS_PANDAS:
    df_plot = df_fact[df_fact['Итого_Дельта_руб'].abs() > 5.0].sort_values('Итого_Дельта_руб', ascending=True)
    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=df_plot['service_name'],
        x=df_plot['Эффект_Объема_руб'],
        name='Фактор объема (расход)',
        orientation='h',
        marker_color='#3498db',
        customdata=df_plot[['v0', 'v1', 't0']].values,
        hovertemplate="<b>%{y}</b><br>Фактор объема: <b>%{x:+,.2f} ₽</b><br>Расход: %{customdata[0]:.2f} ➔ %{customdata[1]:.2f}<br>Тариф базы: %{customdata[2]:,.2f} ₽<extra></extra>"
    ))
    fig.add_trace(go.Bar(
        y=df_plot['service_name'],
        x=df_plot['Эффект_Тарифа_руб'],
        name='Фактор тарифа (индексация цен)',
        orientation='h',
        marker_color='#e67e22',
        customdata=df_plot[['t0', 't1', 'v1']].values,
        hovertemplate="<b>%{y}</b><br>Фактор тарифа: <b>%{x:+,.2f} ₽</b><br>Тариф: %{customdata[0]:,.2f} ➔ %{customdata[1]:,.2f} ₽<br>Фактический объем: %{customdata[2]:.2f}<extra></extra>"
    ))
    fig.update_layout(
        title="<b>Факторный анализ изменений: Сентябрь 2025 vs Сентябрь 2026 (руб.)</b>",
        barmode='group',
        xaxis_title="Вклад в изменение начислений, руб.",
        xaxis_ticksuffix=" ₽",
        legend=dict(orientation="h", yanchor="bottom", y=-0.2, xanchor="center", x=0.5),
        height=520,
        margin=dict(l=220, r=40, t=60, b=60)
    )
    show_fig(fig)
elif HAS_MPL and HAS_PANDAS:
    df_plot = df_fact[df_fact['Итого_Дельта_руб'].abs() > 5.0].sort_values('Итого_Дельта_руб')
    fig, ax = plt.subplots(figsize=(11, 5.5))
    y_pos = range(len(df_plot))
    height = 0.38
    ax.barh([y - height/2 for y in y_pos], df_plot['Эффект_Объема_руб'], height=height, color='#3498db', alpha=0.85, label='Фактор объема (расход ресурса)', edgecolor='black')
    ax.barh([y + height/2 for y in y_pos], df_plot['Эффект_Тарифа_руб'], height=height, color='#e67e22', alpha=0.85, label='Фактор тарифа (индексация цен)', edgecolor='black')
    ax.axvline(0, color='black', linewidth=1, linestyle='-')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot['service_name'], fontsize=9.5)
    ax.set_title("Факторный анализ изменений: Сентябрь 2025 vs Сентябрь 2026 (руб.)", fontsize=13, pad=12)
    ax.set_xlabel("Вклад в изменение начислений, руб.")
    ax.grid(axis='x', linestyle="--", alpha=0.6)
    ax.legend(loc="lower right", frameon=True)
    plt.tight_layout()
    plt.show()
else:
    for r in conn.execute(query_factor):
        print(dict(r))


,Услуга,"Сен 2025, руб.","Сен 2026, руб.","Общее изменение, руб.","Фактор объема, руб.","Фактор тарифа, руб."
0,СОДЕРЖАНИЕ Ж/Ф,2846.58,3093.78,247.20,0.00,247.20
1,ГОРЯЧЕЕ В/С (ЭНЕРГИЯ),2945.38,2073.08,-872.30,-906.29,33.98
2,ВЗНОС НА КАПИТАЛЬНЫЙ РЕМОНТ,1498.20,1498.20,0.00,0.00,0.00
3,ЭЛЕКТРИЧЕСТВО ОДНОТАРИФНЫЙ ПУ (Д1),844.80,1246.20,401.40,382.80,18.60
4,ОБРАЩЕНИЕ С ТКО,771.84,784.71,12.87,0.00,12.87
5,ВОДООТВЕДЕНИЕ,1063.23,722.54,-340.69,-352.49,11.79
6,ГОРЯЧЕЕ В/С (НОСИТЕЛЬ),599.69,413.28,-186.41,-184.52,-1.89
7,ХОЛОДНОЕ В/С,369.04,231.34,-137.70,-136.64,-1.06
8,ЭЛЕКТРОСНАБЖЕНИЕ ОДН,201.39,204.44,3.05,0.00,3.05
9,ДОМОФОН,55.00,65.00,10.00,0.00,10.00


---
## 13. Интерактивный селектор услуг (ipywidgets Dropdown)
Выберите любую услугу из выпадающего списка для детального анализа её истории:
- Динамика расхода (в натуральных единицах) и начислений (в рублях).
- Ступенчатый график изменений тарифа.


In [92]:
# Загружаем историю по всем услугам
query_all_services = """
    SELECT 
        r.period_date,
        r.period,
        sc.service_name,
        sc.volume,
        sc.unit,
        sc.tariff,
        sc.billed_amount,
        sc.total_amount
    FROM service_charges sc
    JOIN receipts r ON sc.receipt_id = r.id
    ORDER BY r.period_date;
"""

if HAS_PANDAS:
    df_all_srv = pd.read_sql_query(query_all_services, conn)
    services_list = sorted(df_all_srv['service_name'].unique())
    
    def render_plotly_service_dashboard(selected_service):
        sub = df_all_srv[df_all_srv['service_name'] == selected_service].copy().sort_values('period_date')
        if sub.empty:
            print("Нет данных по выбранной услуге.")
            return
        unit = sub['unit'].iloc[0] or 'ед.'
        
        fig = make_subplots(
            rows=2, cols=1,
            specs=[[{"secondary_y": True}], [{}]],
            subplot_titles=[
                f"<b>«{selected_service}»: объем потребления и сумма начислений</b>",
                "<b>Динамика утвержденной ставки тарифа</b>"
            ],
            vertical_spacing=0.15
        )
        # 1. Объем (левая Y) и Сумма (правая Y)
        fig.add_trace(
            go.Scatter(
                x=sub['period'],
                y=sub['volume'],
                name=f"Объем ({unit})",
                mode="lines+markers",
                line=dict(color="#2980b9", width=2.5),
                marker=dict(size=7),
                hovertemplate=f"<b>%{{x}}</b><br>Расход: <b>%{{y}} {unit}</b><extra></extra>"
            ),
            row=1, col=1, secondary_y=False
        )
        fig.add_trace(
            go.Scatter(
                x=sub['period'],
                y=sub['total_amount'],
                name="Начислено, ₽",
                mode="lines+markers",
                line=dict(color="#e74c3c", width=2.5),
                marker=dict(size=7, symbol="square"),
                hovertemplate="<b>%{x}</b><br>Сумма: <b>%{y:,.2f} ₽</b><extra></extra>"
            ),
            row=1, col=1, secondary_y=True
        )
        # 2. Ступенчатый тариф
        fig.add_trace(
            go.Scatter(
                x=sub['period'],
                y=sub['tariff'],
                name="Тариф, ₽",
                mode="lines+markers",
                line=dict(color="#27ae60", width=2.5, shape="hv"),
                marker=dict(size=7),
                hovertemplate="<b>%{x}</b><br>Тарифная ставка: <b>%{y:,.4f} ₽</b><extra></extra>"
            ),
            row=2, col=1
        )
        fig.update_layout(
            hovermode="x unified",
            height=680,
            margin=dict(t=60, b=50, l=60, r=60),
            legend=dict(orientation="h", yanchor="bottom", y=-0.15, xanchor="center", x=0.5)
        )
        fig.update_yaxes(title_text=f"Объем, {unit}", secondary_y=False, row=1, col=1)
        fig.update_yaxes(title_text="Сумма, руб.", ticksuffix=" ₽", secondary_y=True, row=1, col=1)
        fig.update_yaxes(title_text="Тариф, руб.", ticksuffix=" ₽", row=2, col=1)
        show_fig(fig)
        
    if HAS_PLOTLY:
        try:
            import ipywidgets as widgets
            from ipywidgets import interact
            print("🎛️ Интерактивный селектор Plotly активен. Выберите услугу из списка:")
            dropdown = widgets.Dropdown(
                options=services_list,
                value='СОДЕРЖАНИЕ Ж/Ф' if 'СОДЕРЖАНИЕ Ж/Ф' in services_list else services_list[0],
                description='Услуга:',
                layout=widgets.Layout(width='450px')
            )
            interact(render_plotly_service_dashboard, selected_service=dropdown)
        except ImportError:
            print("Примечание: ipywidgets не установлен. Демонстрация Plotly для услуги по умолчанию:")
            default_srv = 'СОДЕРЖАНИЕ Ж/Ф' if 'СОДЕРЖАНИЕ Ж/Ф' in services_list else services_list[0]
            render_plotly_service_dashboard(default_srv)
    elif HAS_MPL:
        def render_service_dashboard_mpl(selected_service):
            sub = df_all_srv[df_all_srv['service_name'] == selected_service].copy().sort_values('period_date')
            if sub.empty:
                return
            unit = sub['unit'].iloc[0] or 'ед.'
            x_pos = range(len(sub))
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
            ax1_twin = ax1.twinx()
            l1 = ax1.plot(x_pos, sub['volume'], marker='o', color='#2980b9', linewidth=2, label=f"Объем ({unit})")
            l2 = ax1_twin.plot(x_pos, sub['total_amount'], marker='s', color='#e74c3c', linewidth=2, label="Начислено, руб.")
            ax1.set_title(f"Услуга: «{selected_service}» — объем потребления и начисления", fontsize=13, pad=10)
            ax1.set_ylabel(f"Объем, {unit}", color='#2980b9')
            ax1_twin.set_ylabel("Сумма, руб.", color='#c0392b')
            ax1.grid(True, linestyle="--", alpha=0.5)
            lines = l1 + l2
            labels = [l.get_label() for l in lines]
            ax1.legend(lines, labels, loc="upper left")
            ax2.step(x_pos, sub['tariff'], where='mid', color='#27ae60', linewidth=2.2, label="Тариф, руб.")
            ax2.plot(x_pos, sub['tariff'], 'o', color='#27ae60', markersize=5)
            ax2.set_title("Динамика ставки тарифа", fontsize=12, pad=10)
            ax2.set_ylabel("Тариф, руб.")
            ax2.set_xlabel("Период")
            ax2.set_xticks(x_pos)
            ax2.set_xticklabels(sub['period'], rotation=30)
            ax2.grid(True, linestyle="--", alpha=0.5)
            ax2.legend(loc="upper left")
            plt.tight_layout()
            plt.show()
        default_srv = 'СОДЕРЖАНИЕ Ж/Ф' if 'СОДЕРЖАНИЕ Ж/Ф' in services_list else services_list[0]
        render_service_dashboard_mpl(default_srv)
    else:
        for r in conn.execute(query_all_services):
            print(dict(r))


🎛️ Интерактивный селектор Plotly активен. Выберите услугу из списка:


interactive(children=(Dropdown(description='Услуга:', index=11, layout=Layout(width='450px'), options=('ВЗНОС …